# Behavioural patterns

- [IHandler](#IHandler)
- [ICommand](#ICommand)
- [IInvoker](#IInvoker)
- [IExpression](#IExpression)
- [IIterator](#IIterator)
- [IAsyncIterator](#IAsyncIterator)
- [IAggregate](#IAggregate)
- [IMediator](#IMediator)
- [IColleague](#IColleague)
- [IMemento](#IMemento)
- [IOriginator](#IOriginator)
- [IObserver](#IObserver)
- [IObservable](#IObservable)
- [IState](#IState)
- [IStateContext](#IStateContext)
- [IStrategy](#IStrategy)
- [IStrategyContext](#IStrategyContext)
- [ITemplate](#ITemplate)
- [IVisitor](#IVisitor)
- [IElement](#IElement)
- [ILogger](#ILogger)
- [IObserver](#Observer)

## Observer


In [ ]:
# Install prometheus_client
!pip install prometheus_client

In [ ]:
import time
from typing import List
from abc import ABC, abstractmethod
from prometheus_client import start_http_server, Counter, Gauge
from wattleflow.core import IObservable, IObserver

# For OSCAL: Implementing interfaces IObserver and IObservable

class IObservable(ABC):
    def __init__(self):
        self._observers: List[IObserver] = []

    def register_observer(self, observer: IObserver) -> None:
        self._observers.append(observer)

    def notify_observers(self, event: str, data: dict) -> None:
        for observer in self._observers:
            observer.update(event, data)

# Prometheus Counters for audit metrics
audit_event_counter = Counter('audit_events_total', 'Total number of audit events', ['event_type'])
audit_event_duration = Gauge('audit_event_duration_seconds', 'Duration of audit event processing')

# Concrete implementation of an Audit observer
class AuditLogger(IObserver):
    def update(self, event: str, data: dict) -> None:
        # Log the event and data to console or external service
        print(f"Audit Event: {event}, Data: {data}")
        # Increase the Prometheus counter for the given event type
        audit_event_counter.labels(event_type=event).inc()

# Audit system that acts as the Observable
class AuditSystem(IObservable):
    def log_event(self, event: str, data: dict) -> None:
        # Record how long the event takes to process
        start_time = time.time()
        self.notify_observers(event, data)
        duration = time.time() - start_time
        # Record the duration metric in Prometheus
        audit_event_duration.set(duration)

# Example usage of the audit system
if __name__ == "__main__":
    # Start Prometheus metrics server on port 8000
    start_http_server(8000)

    # Initialize the audit system and register an observer
    audit_system = AuditSystem()
    audit_logger = AuditLogger()
    audit_system.register_observer(audit_logger)

    # Simulate logging of audit events
    audit_system.log_event("user_login", {"user": "john_doe", "status": "success"})
    time.sleep(2)
    audit_system.log_event("file_access", {"file": "/etc/passwd", "user": "john_doe", "action": "read"})

    # Keep the application running to expose metrics
    while True:
        time.sleep(1)
